# lab08 · Agent-Driver 风格的 LLM tool-calling
**配套节点**：[Agent-Driver](../docs/data/cards/paper_2311.10813_agent_driver.md)

**What this proves**：LLM 通过 *function call* 调用一个轻量 perception tool library，
在 mock JSON 上做出端到端的驾驶决策；可拓展到真实感知后端。


In [1]:
import sys, os, json
sys.path.insert(0, "..")
from labs.llm_provider import LLM
llm = LLM()

# Mock perception world
WORLD = {
    "vehicles": [
        {"id": 1, "lon": 12.0, "lat": 0.5, "speed": 18},
        {"id": 2, "lon": 25.0, "lat": -3.5, "speed": 22},
        {"id": 3, "lon": 50.0, "lat": 4.0, "speed": 30},
    ],
    "speed_limit": 27.0,
    "ego": {"speed": 26.0, "lane": 1},
}

# tool library
def get_nearby_vehicles(radius_m=50):
    return [v for v in WORLD["vehicles"] if v["lon"] <= radius_m]
def get_speed_limit(): return WORLD["speed_limit"]
def get_ego_state():    return WORLD["ego"]

TOOLS = {
    "get_nearby_vehicles": get_nearby_vehicles,
    "get_speed_limit":     get_speed_limit,
    "get_ego_state":       get_ego_state,
}
TOOL_SCHEMA = {k: f.__doc__ or k for k, f in TOOLS.items()}


In [2]:
def llm_choose_tool(query):
    sys_msg = ("你是 Agent-Driver。先选择一个 tool 收集信息，再给出 meta_action。"
               "返回 JSON: {\"tool\": <name>, \"args\": <dict>, \"rationale\": <str>}.")
    user = f"可调用 tool: {list(TOOLS)}\n查询: {query}"
    return llm.json_chat([{"role":"system","content":sys_msg},{"role":"user","content":user}])

def llm_finalize(query, ctx):
    sys_msg = ("基于以下 tool 返回的上下文，输出最终 driving decision JSON: "
               "{meta_action, rationale, target_speed, target_lane}.")
    user = f"查询: {query}\n上下文: {json.dumps(ctx, ensure_ascii=False)}"
    return llm.json_chat([{"role":"system","content":sys_msg},{"role":"user","content":user}])

query = "前方 50m 内是否安全提速？"
choice = llm_choose_tool(query)
print("step 1 — tool choice:", choice)
tname = choice.get("tool", "get_nearby_vehicles")
fn = TOOLS[tname]; args = choice.get("args", {})
ctx = {"tool": tname, "result": fn(**(args or {}))}
print("step 2 — tool result:", ctx)
final = llm_finalize(query, ctx)
print("step 3 — final decision:", final)
assert "meta_action" in final, "agent must output a meta_action"
print("PASS — Agent-Driver loop completes with a structured decision")


step 1 — tool choice: {'meta_action': 'maintain', 'rationale': '前方畅通且巡航，保持', 'target_speed': 25.0, 'target_lane': 1}
step 2 — tool result: {'tool': 'get_nearby_vehicles', 'result': [{'id': 1, 'lon': 12.0, 'lat': 0.5, 'speed': 18}, {'id': 2, 'lon': 25.0, 'lat': -3.5, 'speed': 22}, {'id': 3, 'lon': 50.0, 'lat': 4.0, 'speed': 30}]}
step 3 — final decision: {'meta_action': 'maintain', 'rationale': '前方畅通且巡航，保持', 'target_speed': 25.0, 'target_lane': 1}
PASS — Agent-Driver loop completes with a structured decision


### 三个 stretch goals
1. 增加 *get_lane_topology* / *get_traffic_signal* 工具，要求 LLM 多轮调用；
2. 给 LLM 加一个 *reject* 选项："信息不足，请人类接管"；
3. 把 mock WORLD 换成 nuScenes 单帧 GT JSON，复刻论文 §4.2。
